## 👨‍💻 Solutions 6.5: Modern Class Patterns

**Using dataclasses:**

1. Create a dataclass called `Product` with attributes for `name`, `price`, `category`, and `in_stock` (with a default value of True). Include a method called `discount_price` that takes a percentage and returns the discounted price.

2. Create a few product instances and demonstrate the automatically generated `__repr__` and `__eq__` methods.

**Named tuples:**

3. Create a named tuple called `Point3D` with fields for `x`, `y`, and `z` coordinates. Create a few points and demonstrate accessing the coordinates both by name and by index.

**Composition:**

4. Design a system using composition to model a `ShoppingCart` that contains a collection of `Product` items and a `Customer`. The `ShoppingCart` should have methods to add products, remove products, and calculate the total price.

*Using dataclasses:*

In [ ]:
from dataclasses import dataclass

# Task 1: Create a Product dataclass with methods
@dataclass
class Product:
    """A dataclass representing a product in inventory."""
    name: str
    price: float
    category: str
    in_stock: bool = True  # Default value
    
    def discount_price(self, percent):
        """Calculate the discounted price.
        
        Args:
            percent: Discount percentage (0-100)
        
        Returns:
            The price after discount
        """
        # Ensure the discount is valid
        if not 0 <= percent <= 100:
            raise ValueError("Discount must be between 0 and 100")
        
        return self.price * (1 - percent / 100)

In [ ]:
# Task 2: Demonstrate automatically generated methods
# Create some products
product1 = Product("Laptop", 999.99, "Electronics")
product2 = Product("Coffee Mug", 12.99, "Kitchenware")
product3 = Product("Laptop", 999.99, "Electronics")  # Same as product1
product4 = Product("Notebook", 5.99, "Office Supplies", in_stock=False)

# Test the auto-generated __repr__ method
print("Testing __repr__ method:")
print(product1)  # Shows all fields
print(product2)
print(product4)  # Shows the in_stock=False

# Test the auto-generated __eq__ method
print("\nTesting __eq__ method:")
print(f"product1 == product2: {product1 == product2}")  # False
print(f"product1 == product3: {product1 == product3}")  # True (same values)
print(f"product1 is product3: {product1 is product3}")  # False (different objects)

# Test our custom method
print("\nTesting discount_price method:")
print(f"{product1.name} original price: ${product1.price:.2f}")
print(f"{product1.name} with 10% discount: ${product1.discount_price(10):.2f}")
print(f"{product1.name} with 20% discount: ${product1.discount_price(20):.2f}")

*Named tuples:*

In [ ]:
# Task 3: Create and use a named tuple
from typing import NamedTuple

# Create a named tuple class for 3D points
class Point3D(NamedTuple):
    """A named tuple representing a point in 3D space."""
    x: float
    y: float
    z: float
    
    def distance_from_origin(self):
        """Calculate distance from origin (0,0,0)."""
        return (self.x**2 + self.y**2 + self.z**2)**0.5


# Create some points
p1 = Point3D(1.0, 2.0, 3.0)
p2 = Point3D(5.0, 6.0, 7.0)
p3 = Point3D(-2.0, 3.0, 1.0)

# Access by name
print("Accessing by name:")
print(f"p1 coordinates: x={p1.x}, y={p1.y}, z={p1.z}")

# Access by index
print("\nAccessing by index:")
print(f"p2 coordinates: x={p2[0]}, y={p2[1]}, z={p2[2]}")

# Demonstrate method on named tuple
print("\nUsing methods on named tuples:")
for point in [p1, p2, p3]:
    print(f"Distance from origin for {point}: {point.distance_from_origin():.2f}")

# Demonstrate immutability
print("\nDemonstrating immutability:")
try:
    p1.x = 10  # This will raise an error
except AttributeError as e:
    print(f"Error: {e}")

# Unpacking the named tuple
print("\nUnpacking named tuple:")
x, y, z = p2
print(f"Unpacked p2: x={x}, y={y}, z={z}")

*Composition:*

In [ ]:
# Task 4: Implement composition for a shopping cart system
from dataclasses import dataclass
from typing import List, Optional

@dataclass
class Product:
    """Product class from tasks 1-2."""
    name: str
    price: float
    category: str
    in_stock: bool = True
    
    def discount_price(self, percent):
        """Calculate the discounted price."""
        if not 0 <= percent <= 100:
            raise ValueError("Discount must be between 0 and 100")
        return self.price * (1 - percent / 100)


@dataclass
class Customer:
    """A customer with basic information."""
    name: str
    email: str
    address: str
    
    def __str__(self):
        """Override __str__ for nicer display."""
        return f"{self.name} ({self.email})"


class ShoppingCart:
    """A shopping cart that contains products for a customer.
    
    This class demonstrates composition by containing:
    - A Customer object
    - A list of Product objects with quantities
    """
    
    def __init__(self, customer):
        """Initialize shopping cart with a customer and empty item list."""
        # Composition: ShoppingCart HAS-A Customer
        self.customer = customer
        # Composition: ShoppingCart HAS Products
        self.items = []  # List to store products and quantities
    
    def add_product(self, product, quantity=1):
        """Add a product to the cart.
        
        Args:
            product: The Product to add
            quantity: How many to add (default 1)
            
        Returns:
            Tuple of (success, message)
        """
        # Check if product is in stock
        if not product.in_stock:
            return False, f"Sorry, {product.name} is out of stock."
        
        # Check if product is already in cart
        for item in self.items:
            if item["product"] == product:
                item["quantity"] += quantity
                return True, f"Updated quantity of {product.name} to {item['quantity']}."
        
        # If not already in cart, add it
        self.items.append({"product": product, "quantity": quantity})
        return True, f"Added {quantity} {product.name} to cart."
    
    def remove_product(self, product):
        """Remove a product from the cart.
        
        Args:
            product: The Product to remove
            
        Returns:
            Tuple of (success, message)
        """
        for i, item in enumerate(self.items):
            if item["product"] == product:
                del self.items[i]
                return True, f"Removed {product.name} from cart."
        
        return False, f"{product.name} not found in cart."
    
    def calculate_total(self):
        """Calculate the total price of all items in the cart."""
        return sum(item["product"].price * item["quantity"] for item in self.items)
    
    def apply_discount(self, percent):
        """Apply a discount to the entire cart."""
        return sum(item["product"].discount_price(percent) * item["quantity"] 
                  for item in self.items)
    
    def __str__(self):
        """Return a string representation of the cart."""
        if not self.items:
            return f"Shopping cart for {self.customer} is empty."
        
        item_count = sum(item["quantity"] for item in self.items)
        return f"Shopping cart for {self.customer} contains {item_count} items."


# Test the shopping cart system
# Create products
print("Creating products...")
laptop = Product("Laptop", 999.99, "Electronics")
headphones = Product("Headphones", 89.99, "Electronics")
book = Product("Python Cookbook", 39.99, "Books")
out_of_stock_item = Product("Keyboard", 49.99, "Electronics", in_stock=False)

# Create customer and cart
print("\nCreating customer and cart...")
customer = Customer("John Doe", "john@example.com", "123 Main St")
cart = ShoppingCart(customer)
print(cart)  # Empty cart

# Add products
print("\nAdding products to cart...")
success, message = cart.add_product(laptop)
print(message)
success, message = cart.add_product(headphones, 2)
print(message)
success, message = cart.add_product(book)
print(message)
success, message = cart.add_product(out_of_stock_item)
print(message)  # Should show an error

# Show cart contents and total
print("\nCart contents:")
print(cart)
print(f"Total: ${cart.calculate_total():.2f}")
print(f"Total with 10% discount: ${cart.apply_discount(10):.2f}")

# Remove an item
print("\nRemoving an item...")
success, message = cart.remove_product(book)
print(message)
print(f"New total: ${cart.calculate_total():.2f}")

# Try to remove a non-existent item
success, message = cart.remove_product(out_of_stock_item)
print(message)  # Should show item not found